# Smit Patel Contribution


In [ ]:
!pip install pandas datasets langchain faiss-cpu sentence-transformers \
  huggingface_hub transformers accelerate --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 27.1 MB/s eta 0:00:00


In [ ]:
!pip install -U langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from datasets import load_dataset
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch


In [ ]:
print(passages.columns)
passages.head()


Index(['passage'], dtype='object')


,passage
id,
9797,New data on viruses isolated from patients wit...
11906,We describe an improved method for detecting d...
16083,We have studied the effects of curare on respo...
23188,Kinetic and electrophoretic properties of 230-...
23469,Male Wistar specific-pathogen-free rats aged 2...


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


/tmp/ipython-input-10-3291538701.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document

# Define the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Use only the first 5000 passages
passages_subset = passages.iloc[:5000].drop_duplicates(subset='passage')

# Split text into chunks
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Convert to LangChain Document format
documents = [
    Document(page_content=row['passage'], metadata={"source": f"id_{i}"})
    for i, row in passages_subset.iterrows()
]

# Chunk the documents
chunks = text_splitter.split_documents(documents)

# Build FAISS index
db = FAISS.from_documents(chunks, embedding_model)

# Save index
db.save_local("bioasq_faiss_index_5000")


In [ ]:
!pip install llama-cpp-python --prefer-binary


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 MB 10.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.14-cp311-cp311-linux_x86_64.whl size=4237784 sha256=7b050ed584a3488e7792161e6f7951b04cc0f20d3fe6709fec7d3f921d068c38
  Stored in directory: /root/.cache/pip/wheels/3f/b6/cf/7315ec7b0149210d2d4447d9c3338b36d10e56a1ecddcd35c0
Successfully built llama-cpp-python


In [ ]:
from google.colab import files
import os

# Step 1: Upload files
uploaded = files.upload()




Saving tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf to tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
✅ File uploaded: tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
📁 Full path: /content/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


In [ ]:
from langchain.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/content/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    n_ctx=2048,
    n_batch=256,
    f16_kv=True,
    n_gpu_layers=0,   # Set >0 if using GPU
    verbose=True
)


llama_model_loader: loaded meta data with 23 key-value pairs and 201 tensors from /content/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = tinyllama_tinyllama-1.1b-chat-v1.0
llama_model_loader: - kv   2:                       llama.context_length u32              = 2048
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 5632
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 64
llama_model_loader: - kv   7:            

In [ ]:
response = llm.invoke("What is Retrieval-Augmented Generation?")
print(response)


llama_perf_context_print:        load time =    1354.29 ms
llama_perf_context_print: prompt eval time =    1353.75 ms /    12 tokens (  112.81 ms per token,     8.86 tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /     1 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =    1356.03 ms /    13 tokens


In [ ]:
from langchain.vectorstores import FAISS

# Load FAISS index safely (since you created it)
db = FAISS.load_local(
    "/content/bioasq_faiss_index_5000",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)


In [ ]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})


In [ ]:
import langchain
print(langchain.__version__)


0.3.26


In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.vectorstores import FAISS

# Load the FAISS index (you already created this earlier)
from langchain.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.load_local("/content/bioasq_faiss_index_5000", embeddings=embedding_model, allow_dangerous_deserialization=True)

# Create a retriever from the FAISS index
retriever = db.as_retriever()

# Set up conversation memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Now use the LlamaCpp LLM you've already loaded earlier
from langchain.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/content/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
    n_ctx=2048,
    temperature=0.7,
    top_p=0.95,
    verbose=True
)

# ✅ Build the Conversational Retrieval Chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)


llama_model_loader: loaded meta data with 23 key-value pairs and 201 tensors from /content/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = tinyllama_tinyllama-1.1b-chat-v1.0
llama_model_loader: - kv   2:                       llama.context_length u32              = 2048
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 2048
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 5632
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 64
llama_model_loader: - kv   7:            

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 2})

# Rebuild the chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    verbose=True
)


In [ ]:
question = "What is the function of dopamine in the brain?"
response = rag_chain.invoke({"question": question})
print(response["answer"])




> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

1. (-)-Deprenyl has been shown to potentiate rat striatal neurone responses to 
dopamine agonists at doses not altering dopamine metabolism. Since there are a 
number of effects of (-)-deprenyl which could result in this phenomenon, we have 
investigated the effects of MDL 72,145 and Ro 19-6327, whose only common effect 
with (-)-deprenyl is an inhibition of monoamine oxidase-B (MAO-B), on rat 
striatal neurone responses to dopamine and on striatal dopamine metabolism. 2. 
Using in vivo electrophysiology, i.p. injection of either MDL 72,145 or Ro 
19-6327 was found to produce a dose-dependent potentiation of striatal neurone 
responses to dopamine but not gamma-aminobutyric acid. 3. Neurochemical 
investigations revealed

Llama.generate: 4 prefix-match hit, remaining 1525 prompt tokens to eval
llama_perf_context_print:        load time =   39174.22 ms
llama_perf_context_print: prompt eval time =  157756.96 ms /  1526 tokens (  103.38 ms per token,     9.67 tokens per second)
llama_perf_context_print:        eval time =   16636.82 ms /   111 runs   (  149.88 ms per token,     6.67 tokens per second)
llama_perf_context_print:       total time =  124398.40 ms /  1637 tokens



> Finished chain.

> Finished chain.
 Dopamin is a neurotransmitter that acts on the neuromuscular 
system, specifically the dopamine receptors on motor neurons. It is involved in 
the control of movement and the regulation of feeding behavior. In Parkinson's 
Disease, it has been suggested to be a neurodegenerative disease involving a 
loss of dopaminreceptor density in the substantia nigra, which contributes to the 
onset of motor symptoms.


First prompt: Explain Rag model
Last Prompt: How can I solve version mismatch